In [1]:
from pathlib import Path

# ===== AJUSTE AQUI =====
REAL_ROOT       = Path("DATASET54000")
SYNTH_ORIG_ROOT = Path("DDPM-F64x4")
SYNTH_FILT_ROOT = Path("DDPM-FILTERED-KNN-54K")

BATCH_SIZE = 128

print("REAL_ROOT:", REAL_ROOT)
print("SYNTH_ORIG_ROOT:", SYNTH_ORIG_ROOT)
print("SYNTH_FILT_ROOT:", SYNTH_FILT_ROOT)


REAL_ROOT: DATASET54000
SYNTH_ORIG_ROOT: DDPM-F64x4
SYNTH_FILT_ROOT: DDPM-FILTERED-KNN-54K


In [2]:
import numpy as np
from PIL import Image
from typing import List, Dict, Tuple

import torch
from torchvision import transforms
from torchmetrics.image.fid import FrechetInceptionDistance

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}

# Backbone EXACTAMENTE igual ao filtro 64
fid_metric = FrechetInceptionDistance(feature=2048, normalize=True).to(DEVICE)
fid_metric.eval()

if hasattr(fid_metric, "inception"):
    inception_net = fid_metric.inception
elif hasattr(fid_metric, "feature_network"):
    inception_net = fid_metric.feature_network
else:
    raise RuntimeError("Não achei a rede Inception dentro do torchmetrics FID.")

inception_net.eval().to(DEVICE)

to_uint8_64 = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),      # 0..1
])


DEVICE: cuda


In [3]:
def list_classes(root: Path) -> List[str]:
    classes = [d.name for d in root.iterdir() if d.is_dir()]
    classes.sort()
    return classes

def list_images(class_dir: Path) -> List[Path]:
    paths = []
    for p in class_dir.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            paths.append(p)
    paths.sort()
    return paths

def batched(lst: List[Path], bs: int):
    for i in range(0, len(lst), bs):
        yield lst[i:i+bs]


In [4]:
@torch.no_grad()
def extract_features_64(paths: List[Path], batch_size: int) -> np.ndarray:
    feats_all = []

    for chunk in batched(paths, batch_size):
        batch = []
        for p in chunk:
            img = Image.open(p).convert("RGB")
            t = to_uint8_64(img)                     # [3,64,64] float
            t = (t * 255.0).clamp(0, 255).byte()     # uint8
            batch.append(t)

        x = torch.stack(batch, dim=0).to(DEVICE)    # [B,3,64,64] uint8
        feats = inception_net(x)
        if isinstance(feats, (tuple, list)):
            feats = feats[0]

        feats_all.append(feats.detach().cpu().numpy().astype(np.float64))

    if not feats_all:
        return np.zeros((0, 2048), dtype=np.float64)

    return np.vstack(feats_all)


In [5]:
def compute_stats(feats: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    mu = np.mean(feats, axis=0)
    sigma = np.cov(feats, rowvar=False)
    return mu, sigma


def frechet_distance(mu1, sigma1, mu2, sigma2, eps=1e-6) -> float:
    from scipy import linalg

    diff = mu1 - mu2
    covmean = linalg.sqrtm(sigma1 @ sigma2)

    if np.iscomplexobj(covmean):
        covmean = covmean.real

    if not np.isfinite(covmean).all():
        offset = np.eye(sigma1.shape[0]) * eps
        covmean = linalg.sqrtm((sigma1 + offset) @ (sigma2 + offset))
        if np.iscomplexobj(covmean):
            covmean = covmean.real

    fid = (
        diff.dot(diff)
        + np.trace(sigma1)
        + np.trace(sigma2)
        - 2.0 * np.trace(covmean)
    )
    return float(fid)


In [6]:
import json

classes = sorted(
    set(list_classes(REAL_ROOT))
    & set(list_classes(SYNTH_ORIG_ROOT))
    & set(list_classes(SYNTH_FILT_ROOT))
)

print("Classes:", classes)

results: Dict[str, Dict] = {}

for cls in classes:
    real_paths = list_images(REAL_ROOT / cls)
    orig_paths = list_images(SYNTH_ORIG_ROOT / cls)
    filt_paths = list_images(SYNTH_FILT_ROOT / cls)

    if len(real_paths) < 2 or len(orig_paths) < 2 or len(filt_paths) < 2:
        print(f"[SKIP] {cls}: imagens insuficientes")
        continue

    feats_real = extract_features_64(real_paths, BATCH_SIZE)
    feats_orig = extract_features_64(orig_paths, BATCH_SIZE)
    feats_filt = extract_features_64(filt_paths, BATCH_SIZE)

    mu_r, s_r = compute_stats(feats_real)
    mu_o, s_o = compute_stats(feats_orig)
    mu_f, s_f = compute_stats(feats_filt)

    fid_orig = frechet_distance(mu_r, s_r, mu_o, s_o)
    #fid_filt = frechet_distance(mu_r, s_r, mu_f, s_f)

    results[cls] = {
        "n_real": len(real_paths),
        "n_orig": len(orig_paths),
        #"n_filt": len(filt_paths),
        "FID_real_vs_orig_64": fid_orig,
        #"FID_real_vs_filt_64": fid_filt,
        #"delta_orig_minus_filt": fid_orig - fid_filt,
        #"improved": fid_filt < fid_orig,
    }

    print(
        f"{cls:>18} | "
        f"FID orig={fid_orig:.3f} | "
        #f"FID filt={fid_filt:.3f} | "
        #f"Δ={fid_orig - fid_filt:+.3f}"
    )

OUT_JSON = Path("./fid_64_report_by_class.json")
with open(OUT_JSON, "w", encoding="utf-8") as f:
    json.dump({"by_class": results}, f, ensure_ascii=False, indent=2)

print("\nRelatório salvo em:", OUT_JSON.resolve())


Classes: ['BULKCARRIER', 'CONTAINERSHIP', 'GENERALCARGO', 'OILPRODUCTSTANKER', 'PASSENGERSSHIP', 'TANKER', 'TRAWLER', 'TUG', 'VEHICLESCARRIER', 'YACHT']


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/conv.py:459: UserWarning: Applied workaround for CuDNN issue, install nvrtc.so (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:80.)
  return F.conv2d(input, weight, bias, self.stride,


       BULKCARRIER | FID orig=43.010 | 
     CONTAINERSHIP | FID orig=46.700 | 
      GENERALCARGO | FID orig=48.145 | 
 OILPRODUCTSTANKER | FID orig=39.481 | 
    PASSENGERSSHIP | FID orig=33.199 | 
            TANKER | FID orig=46.782 | 
           TRAWLER | FID orig=35.434 | 
               TUG | FID orig=22.407 | 
   VEHICLESCARRIER | FID orig=69.633 | 
             YACHT | FID orig=19.927 | 

Relatório salvo em: /tf/2025/fid_64_report_by_class.json


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plot_classes = list(results.keys())
fid_orig = np.array([results[c]["FID_real_vs_orig_64"] for c in plot_classes])
fid_filt = np.array([results[c]["FID_real_vs_filt_64"] for c in plot_classes])

x = np.arange(len(plot_classes))
width = 0.38

plt.figure(figsize=(15, 6))
plt.bar(x - width/2, fid_orig, width, label="FID 64 — sem filtro")
plt.bar(x + width/2, fid_filt, width, label="FID 64 — com filtro")

plt.xticks(x, plot_classes, rotation=35, ha="right")
plt.ylabel("FID (menor é melhor)")
plt.title("FID por classe (64×64, mesmo backbone do filtro)")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
delta = fid_orig - fid_filt  # positivo = melhora

plt.figure(figsize=(15, 5))
plt.bar(plot_classes, delta)
plt.axhline(0, linewidth=1)

plt.xticks(rotation=35, ha="right")
plt.ylabel("ΔFID = FID(orig) − FID(filt)")
plt.title("Variação de FID por classe após filtragem (64×64)")
plt.tight_layout()
plt.show()

print("Média FID orig:", float(np.mean(fid_orig)))
print("Média FID filt:", float(np.mean(fid_filt)))
print("Média ΔFID:", float(np.mean(delta)))
